# 🏟️ Fine-tuning de **Gemma 4 E2B** — Asistente de reservas con herramientas

Este cuaderno entrena un modelo **Gemma 4 E2B** para que sea el asistente de
reservas del *Polideportivo Municipal Las Encinas*. El modelo aprende a:

- **conversar** de forma natural y pedir los datos que faltan,
- **llamar a herramientas** (`consultar_disponibilidad`, `crear_reserva`,
  `consultar_reserva`, `modificar_reserva`, `cancelar_reserva`),
- **confirmar** antes de cualquier acción que escribe,
- y **reconducir** siempre hacia el objetivo: reservar o consultar una reserva.

### ¿Qué es esto de "fine-tuning con QLoRA"?
- **Fine-tuning** = seguir entrenando un modelo ya hecho con *nuestros* ejemplos
  para especializarlo en una tarea.
- **LoRA** = en vez de tocar los miles de millones de pesos del modelo, añadimos
  unas matrices pequeñas y entrenamos *solo esas*. Mucho más rápido y barato.
- **QLoRA** = LoRA + el modelo base cargado en **4 bits** (ocupa ~4× menos
  memoria). Así cabe en la **GPU T4 gratuita** de Colab.

> ⚙️ **Antes de empezar:** ve a `Entorno de ejecución → Cambiar tipo de entorno`
> y elige **GPU (T4)**. Sin GPU esto no funciona.

**Tiempo aproximado:** 10 min de setup + 30-45 min de entrenamiento.

## 📚 Conceptos clave (léelo una vez)

**Modelo base vs. *instruct*.** Un modelo "base" solo predice la siguiente
palabra. Uno *instruct* (sufijo `-it`) ya fue entrenado para seguir instrucciones
y conversar. Partimos del *instruct* y lo **especializamos** en nuestro dominio.

**Tokens.** El modelo no ve letras ni palabras, sino *tokens* (trozos de texto,
~¾ de palabra de media). El contexto y sus límites se miden en tokens, no en
caracteres.

**Atención (en una frase).** Es el mecanismo que deja a cada token "mirar" a los
demás para entender el contexto; es el corazón del Transformer. Nosotros no lo
tocamos: solo le **enganchamos** unas matrices LoRA.

**¿Por qué fine-tuning y no solo un buen prompt?** Un prompt largo funciona, pero
(1) ocupa contexto en cada petición, (2) es frágil y (3) un modelo pequeño no
siempre lo obedece. El fine-tuning mete el comportamiento **en los pesos**:
respuestas más consistentes y prompts más cortos.

**Cuantización a 4 bits (la "Q" de QLoRA).** Cada peso suele ocupar 16 bits; en
4 bits ocupa 4× menos. E2B (~5,1B pesos con embeddings): en 16 bits ≈ 10 GB, en
4 bits ≈ 3 GB → cabe en la T4. La pérdida de calidad es pequeña y el adapter LoRA
la compensa.

**LoRA (Low-Rank Adaptation).** En vez de modificar una matriz de pesos `W`
(enorme), aprendemos dos matrices pequeñas `A` y `B` cuyo producto `B·A` es el
cambio (de *rango bajo* `r`). Entrenas **<1 %** de los parámetros y el resultado
pesa pocos MB. Conceptos:
- **`r` (rango):** cuántas "dimensiones de cambio" permites (8-32 típico). Más = más
  capacidad y más memoria.
- **`lora_alpha`:** escala del cambio. Regla práctica: `alpha = r`.
- **target modules:** a qué matrices se aplica LoRA → las de **atención**
  (`q,k,v,o_proj`) y las del **MLP** (`gate,up,down_proj`).

**bf16 / fp16.** Formatos de 16 bits para los cálculos. La T4 usa fp16; las GPU
nuevas, bf16. Unsloth elige el adecuado por ti.

## Paso 0 · Comprobar que tenemos GPU
Si esto da error o no muestra una *Tesla T4*, cambia el entorno a GPU (ver arriba).

In [ ]:
!nvidia-smi

## Paso 1 · Instalar **Unsloth**

[Unsloth](https://unsloth.ai) es una librería que hace el fine-tuning **2× más
rápido** y con **menos memoria**, y trae soporte para Gemma 4. Instala también
`transformers`, `trl`, `peft`, etc. en versiones compatibles.

> Si en el futuro algo falla por versiones, revisa la guía oficial de Gemma 4 en
> Unsloth: https://unsloth.ai/docs/models/gemma-4

In [ ]:
%%capture
# La primera vez tarda un par de minutos.
!pip install unsloth

## Paso 2 · Traer nuestro dataset y el backend simulado

Necesitamos tres cosas de la carpeta `fine/`:
- `data/dataset_train.jsonl` y `data/dataset_val.jsonl` → las conversaciones,
- `data/backend_sim.py` → el "backend falso" que usaremos para **probar** el
  modelo ejecutando de verdad sus llamadas a herramientas.

**Opción A (recomendada):** clonar el repositorio (debe estar publicado en GitHub
con la carpeta `fine/`). **Opción B:** subir los ficheros a mano (celda comentada).

In [ ]:
# --- Opción A: clonar el repositorio ---
!git clone https://github.com/noelserdna/capa1-ingesta-colab.git
%cd capa1-ingesta-colab/fine

RUTA_DATOS = "data"
import sys
sys.path.append("data")     # para poder hacer "import backend_sim"

# --- Opción B (si no está publicado): súbelos a mano ---
# from google.colab import files
# print("Sube backend_sim.py, dataset_train.jsonl y dataset_val.jsonl")
# subidos = files.upload()
# RUTA_DATOS = "."
# sys.path.append(".")

import os
print("Ficheros en", RUTA_DATOS, ":", os.listdir(RUTA_DATOS))

## Paso 3 · Cargar **Gemma 4 E2B** en 4 bits

Usamos `FastModel`, el cargador de Unsloth para los modelos Gemma (incluye los
multimodales 3n/4). Lo cargamos en **4 bits** (`load_in_4bit=True`) para que quepa
en la T4.

- `model_name`: la versión *instruct* (`-it`) ya sabe conversar; nosotros solo la
  especializamos. Unsloth ofrece también una versión preparada en 4 bits
  (`...-unsloth-bnb-4bit`) que descarga más rápido.
- `max_seq_length`: longitud máxima de una conversación en tokens. Nuestras
  conversaciones son cortas; **2048** sobra.

In [ ]:
from unsloth import FastModel

MODELO = "unsloth/gemma-4-E2B-it"   # alternativa rápida: "unsloth/gemma-4-E2B-it-unsloth-bnb-4bit"
MAX_SEQ_LEN = 2048

model, tokenizer = FastModel.from_pretrained(
    model_name = MODELO,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit = True,        # QLoRA: base en 4 bits
    full_finetuning = False,    # no tocamos todos los pesos
)

## Paso 4 · Añadir los **adaptadores LoRA**

Aquí "abrimos" el modelo para entrenar solo las matrices LoRA. Parámetros clave:
- **`r`** (rango): tamaño de las matrices LoRA. Más grande = más capacidad de
  aprender, pero más memoria. **16** va bien para un dominio acotado.
- **`lora_alpha`**: escala de la actualización (regla habitual: igual a `r`).
- **`lora_dropout=0`** y **`bias="none"`**: lo óptimo (y más rápido) según Unsloth.
- `finetune_vision_layers=False`: no tocamos la parte de imagen/audio; solo texto.

> Al ejecutar la celda, Unsloth imprime cuántos parámetros vas a entrenar. Verás
> algo tipo *"trainable params: ~20M / 5B (≈0,4 %)"*. **Esa** es la magia de LoRA:
> entrenar una fracción minúscula del modelo.

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers   = False,   # solo texto
    finetune_language_layers = True,
    finetune_attention_modules = True,
    finetune_mlp_modules     = True,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 42,
)

## Paso 5 · El **formato de chat y de herramientas**

El modelo necesita un formato fijo para distinguir quién habla. Gemma usa turnos
`<start_of_turn>user … <end_of_turn>` y `<start_of_turn>model … <end_of_turn>`.

Nuestro caso tiene 4 roles (system, user, assistant, **tool**), así que definimos
una **plantilla de chat propia** que:
- mete el `system` al principio del primer turno de usuario (convención de Gemma),
- pinta `user` y `tool` como turnos de *usuario* (el resultado de la herramienta
  llega "desde fuera", como una entrada más, envuelto en un bloque `tool_result`),
- pinta `assistant` como turno de *model* (ahí van tanto el texto como las
  llamadas `tool_call`).

Como cada `tool_call` va seguido de su `tool_result`, los turnos se **alternan**
perfectamente user/model, que es justo lo que Gemma espera.

In [ ]:
# Plantilla Jinja propia (compatible con Gemma) que entiende el rol "tool".
PLANTILLA = (
    "{{ bos_token }}"
    "{% set ns = namespace(system='') %}"
    "{% for message in messages %}"
        "{% if message['role'] == 'system' %}"
            "{% set ns.system = message['content'] %}"
        "{% elif message['role'] in ['user', 'tool'] %}"
            "{{ '<start_of_turn>user\n' }}"
            "{% if ns.system %}{{ ns.system + '\n\n' }}{% set ns.system = '' %}{% endif %}"
            "{% if message['role'] == 'tool' %}"
                "{{ '```tool_result\n' + message['content'] + '\n```' }}"
            "{% else %}"
                "{{ message['content'] }}"
            "{% endif %}"
            "{{ '<end_of_turn>\n' }}"
        "{% elif message['role'] == 'assistant' %}"
            "{{ '<start_of_turn>model\n' + message['content'] + '<end_of_turn>\n' }}"
        "{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '<start_of_turn>model\n' }}{% endif %}"
)
tokenizer.chat_template = PLANTILLA

# Veamos cómo queda UNA conversación renderizada:
import json
with open(f"{RUTA_DATOS}/dataset_val.jsonl", encoding="utf-8") as f:
    ejemplo = json.loads(f.readline())
print(tokenizer.apply_chat_template(ejemplo["messages"], tokenize=False, add_generation_prompt=False))

## Paso 6 · Cargar y formatear el dataset

Convertimos cada conversación en un único texto con la plantilla anterior. Ese
texto (campo `text`) es lo que verá el entrenador.

In [ ]:
import json
from datasets import Dataset

def cargar_jsonl(ruta):
    with open(ruta, encoding="utf-8") as f:
        filas = [json.loads(l) for l in f]
    # Aplicamos la plantilla a cada conversación -> texto plano.
    textos = [tokenizer.apply_chat_template(r["messages"], tokenize=False,
                                            add_generation_prompt=False) for r in filas]
    return Dataset.from_list([{"text": t} for t in textos])

ds_train = cargar_jsonl(f"{RUTA_DATOS}/dataset_train.jsonl")
ds_val   = cargar_jsonl(f"{RUTA_DATOS}/dataset_val.jsonl")
print("Entrenamiento:", len(ds_train), "| Validación:", len(ds_val))

# Comprobar que ninguna conversación se pasa de MAX_SEQ_LEN tokens.
largos = [len(tokenizer(t["text"]).input_ids) for t in ds_train]
print(f"Tokens por conversación -> media {sum(largos)//len(largos)}, máx {max(largos)}")
assert max(largos) <= MAX_SEQ_LEN, "Sube MAX_SEQ_LEN: alguna conversación se corta."

## Paso 7 · Entrenar (QLoRA)

Configuramos el entrenador (`SFTTrainer` de TRL) y, **muy importante**, usamos
`train_on_responses_only`: así el modelo solo se "penaliza" por lo que dice el
**asistente** (turnos `model`), no por copiar lo que escribe el usuario.

Hiperparámetros (con comentario de qué hace cada uno):
- **`num_train_epochs=2`**: pasadas completas al dataset. Con 1.080 ejemplos, 2-3
  está bien; demasiadas → *overfitting* (memoriza en vez de generalizar).
- **`learning_rate=2e-4`**: cuánto se ajusta en cada paso. Para LoRA suele ir entre
  `1e-4` y `3e-4`. Muy alto → inestable; muy bajo → no aprende.
- **`per_device_train_batch_size=2` × `gradient_accumulation_steps=4`** = **lote
  efectivo 8**. Acumular gradientes simula un lote grande sin gastar más memoria
  (clave en una T4).
- **`warmup_steps=5`**: empieza con un LR pequeño y lo sube; evita "sacudidas" al
  principio.
- **`optim="adamw_8bit"`**: el optimizador en 8 bits → menos memoria.
- **`weight_decay=0.01`**: regularización suave para no sobreajustar.
- **`lr_scheduler_type="linear"`**: el LR baja de forma lineal hasta el final.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds_train,
    eval_dataset = ds_val,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ_LEN,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
        report_to = "none",
    ),
)

# Entrenar SOLO sobre los turnos del asistente (lo que el modelo debe aprender a decir).
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

trainer.train()

### 📉 Cómo leer el entrenamiento

Mientras entrena verás la **`training loss`** (pérdida) cada 10 pasos. Es una medida
de "cuánto se equivoca" prediciendo el siguiente token de los turnos del asistente.

- **Debe ir bajando** y luego aplanarse. El valor absoluto importa poco; lo que
  importa es la **tendencia**.
- Si la `eval loss` (sobre validación) empieza a **subir** mientras la de
  entrenamiento sigue bajando → **overfitting**: baja epochs o `r`.
- Si la loss se queda **alta y plana** desde el principio → quizá el LR es muy bajo,
  o necesitas más epochs / más `r` / más datos.
- Para reservas (dominio acotado) basta con que aprenda el **formato de las
  herramientas** y el **estilo**; no busques una loss "perfecta".

## Paso 8 · Probar el modelo con el **bucle de herramientas**

Aquí está la magia: el modelo genera un turno; si pide una herramienta
(`tool_call`), **ejecutamos esa herramienta de verdad** contra `backend_sim` y le
devolvemos el resultado; repetimos hasta que el modelo responde al usuario.

Es exactamente el bucle que tendrías en producción (solo que con un backend
simulado en lugar de tu base de datos real).

**Parámetros de generación** (en `generar_turno`):
- **`temperature=0.3`**: baja a propósito. Para llamar herramientas y rellenar JSON
  queremos respuestas **predecibles**, no creativas. (Para charla informal subirías
  a ~0.7).
- **`top_p=0.95`**: muestreo de núcleo; descarta la cola improbable.
- **`max_new_tokens=256`**: tope de longitud de cada turno.
- Paramos en **`<end_of_turn>`**, el token con el que Gemma cierra cada turno.

In [ ]:
import re, json
import backend_sim as B

FastModel.for_inference(model)   # modo inferencia (más rápido)
FIN = tokenizer.convert_tokens_to_ids("<end_of_turn>")

def extraer_tool_call(texto):
    """Si el turno del modelo es una llamada a herramienta, devuelve {tool, args}."""
    m = re.search(r"```tool_call\s*(\{.*?\})\s*```", texto, re.DOTALL)
    if not m:
        return None
    try:
        return json.loads(m.group(1))
    except json.JSONDecodeError:
        return None

def generar_turno(messages, max_new_tokens=256):
    entrada = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(entrada, return_tensors="pt", add_special_tokens=False).to("cuda")
    salida = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.3,
                            top_p=0.95, do_sample=True,
                            eos_token_id=[tokenizer.eos_token_id, FIN])
    texto = tokenizer.decode(salida[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
    return texto.split("<end_of_turn>")[0].strip()

def conversar(backend, entradas_usuario):
    """Simula una conversación; backend ejecuta las herramientas que pida el modelo."""
    messages = [{"role": "system", "content": B.system_prompt(backend.hoy)}]
    for texto in entradas_usuario:
        print(f"\n🧑 {texto}")
        messages.append({"role": "user", "content": texto})
        for _ in range(6):   # como mucho 6 llamadas a herramienta por turno
            salida = generar_turno(messages)
            messages.append({"role": "assistant", "content": salida})
            tc = extraer_tool_call(salida)
            if tc is None:
                print(f"🤖 {salida}")
                break
            resultado = backend.ejecutar(tc["tool"], tc.get("args", {}))
            print(f"   🔧 {tc['tool']}({json.dumps(tc.get('args', {}), ensure_ascii=False)})"
                  f" → {json.dumps(resultado, ensure_ascii=False)}")
            messages.append({"role": "tool", "content": json.dumps(resultado, ensure_ascii=False)})
    return messages

# Demo: una reserva de pádel (el usuario va respondiendo)
bk = B.Backend(hoy="2026-09-01", seed=123)
_ = conversar(bk, [
    "Hola, quiero reservar una pista de pádel para mañana por la tarde",
    "a las 7, una hora",
    "seremos 4",
    "Andrés López",
    "sí",
])

Prueba también un caso difícil y uno fuera de tema:

In [ ]:
# Consultar el estado de una reserva que creamos antes
bk2 = B.Backend(hoy="2026-09-01", seed=7)
r = bk2.crear_reserva("tenis", "2026-09-05", "11:00", 60, "Marta Ruiz", num_personas=2)
print("(Reserva sembrada:", r["localizador"], ")")
_ = conversar(bk2, [
    "oye, ¿sabes si mañana va a llover?",          # fuera de tema -> debe reconducir
    "ah vale, pues mira mi reserva de tenis",       # consultar estado
    "por nombre, Marta Ruiz",
])

## Paso 9 · Guardar el resultado

En Colab la sesión es **efímera**: si no guardas, lo pierdes. Guardamos el
**adapter LoRA** (es ligero, unos MB) en tu **Google Drive** para reutilizarlo en
otra sesión sin reentrenar.

In [ ]:
# Guardar solo el adapter LoRA + el tokenizer
model.save_pretrained("lora_gemma4_reservas")
tokenizer.save_pretrained("lora_gemma4_reservas")

# Copiar a Google Drive para que persista
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p "/content/drive/MyDrive/gemma4_reservas"
!cp -r lora_gemma4_reservas "/content/drive/MyDrive/gemma4_reservas/"
print("Adapter guardado en tu Drive: MyDrive/gemma4_reservas/lora_gemma4_reservas")

### (Opcional) Exportar a **GGUF** para Ollama

GGUF es el formato que usa Ollama/llama.cpp para servir el modelo en CPU (tu
futuro servidor de 12 GB). `q4_k_m` es una cuantización equilibrada (~3-4 GB).

> ⚠️ Gemma 4 es muy reciente; si el export GGUF falla por versiones, guarda el
> modelo *fusionado en 16 bits* (`merged_16bit`) y conviértelo con `llama.cpp`, o
> sigue la guía de Unsloth para Gemma 4.

In [ ]:
try:
    model.save_pretrained_gguf("gemma4-e2b-reservas-gguf", tokenizer,
                               quantization_method="q4_k_m")
    print("✅ GGUF exportado en gemma4-e2b-reservas-gguf/")
except Exception as e:
    print("⚠️ Export GGUF falló:", e)
    print("Plan B: guardar fusionado en 16 bits y convertir con llama.cpp.")
    model.save_pretrained_merged("gemma4-e2b-reservas-merged", tokenizer,
                                 save_method="merged_16bit")

## Paso 10 · (Futuro) Llevarlo a **Ollama** en tu servidor

Cuando tengas el `.gguf`, en tu servidor de 12 GB:

1. Copia el `.gguf` y crea un `Modelfile` (lo generamos abajo). El `SYSTEM` debe
   ser **el mismo** system prompt con el que entrenamos (`backend_sim.system_prompt`).
2. `ollama create reservas -f Modelfile`
3. `ollama run reservas`

Tu aplicación tendrá que hacer el **mismo bucle de herramientas** del Paso 8:
detectar el bloque `tool_call`, ejecutar la función contra tu base de datos real,
y devolver el `tool_result`.

In [ ]:
import backend_sim as B

system = B.system_prompt("{{FECHA_DE_HOY}}")   # en producción, pon la fecha real cada día
modelfile = f"""FROM ./gemma4-e2b-reservas-gguf/unsloth.Q4_K_M.gguf
PARAMETER temperature 0.3
PARAMETER num_ctx 8192
PARAMETER stop "<end_of_turn>"
SYSTEM \"\"\"{system}\"\"\"
"""
with open("Modelfile", "w", encoding="utf-8") as f:
    f.write(modelfile)
print(modelfile[:600], "...")
print("\\nEn el servidor:  ollama create reservas -f Modelfile  &&  ollama run reservas")

---
## ✅ Resumen

Has entrenado un Gemma 4 E2B que conversa, **llama a herramientas** y reconduce
siempre hacia la reserva. El adapter LoRA está en tu Drive.

**Siguientes pasos posibles:**
- Generar más datos (`python data/generar_dataset.py --n 3000`) y reentrenar.
- Ajustar el catálogo en `data/backend_sim.py` a tus instalaciones reales.
- Montar el servidor con Ollama (carpeta `fine/deploy/`).